# 02 — Cleaning

This notebook loads the concatenated DVF CSV data, applies the project cleaning pipeline, and saves a cleaned sample for later work.

In [1]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))

from src.data_cleaning import clean_dvf, missing_report, save_cleaning_outputs

DATA_DIR = root / 'data' / 'data_csv'
years = range(2021, 2026)
frames = [pd.read_csv(DATA_DIR / f'ValeursFoncieres-{y}.csv', sep='|', decimal=',', low_memory=False) for y in years]
raw = pd.concat(frames, ignore_index=True)
raw.columns = raw.columns.str.strip()

print('data root:', root)
print('loaded rows:', f'{len(raw):,}')
raw.head()


data root: /Users/arman/Documents/Project/property-sales-pipeline
loaded rows: 3,377,966


,no_disposition,date_mutation,nature_mutation,valeur_fonciere,no_voie,type_de_voie,code_voie,voie,code_postal,commune,code_departement,code_commune,section,no_plan,1er_lot,nombre_de_lots,code_type_local,type_local,surface_reelle_bati,nombre_pieces_principales,nature_culture,surface_terrain
0,1,2021-01-04,Vente,245000.0,12.0,RUE,0042,DU GENERAL DE GAULLE,75001.0,PARIS,75,75001,A,101,NaN,1,2.0,Appartement,48.0,2.0,NaN,NaN
1,2,2021-01-05,Vente,380000.0,7.0,RUE,0017,DE LA PAIX,69001.0,LYON,69,69001,B,204,NaN,0,1.0,Maison,92.0,4.0,S,310.0
2,3,2021-01-06,Vente,155000.0,3.0,AVE,0011,DES FLEURS,13001.0,MARSEILLE,13,13001,C,311,NaN,1,2.0,Appartement,35.0,1.0,NaN,NaN
3,4,2021-01-07,Vente,520000.0,45.0,BLVD,0088,HAUSSMANN,75009.0,PARIS,75,75009,A,509,NaN,2,2.0,Appartement,72.0,3.0,NaN,NaN
4,5,2021-01-08,Vente,210000.0,NaN,NaN,B044,LES CHATAIGNIERS,33000.0,BORDEAUX,33,33000,AC,712,NaN,0,NaN,NaN,NaN,NaN,J,450.0


## Apply cleaning pipeline

Call `clean_dvf` which deduplicates, filters to *Vente* transactions, casts types, and removes invalid values.

In [2]:
cleaned, summary = clean_dvf(raw)
print(f'Rows after cleaning: {len(cleaned):,}')
summary


Rows after cleaning: 2,303,841


,step,rows_before,rows_after,rows_dropped
0,raw input,3377966,3377966,0
1,drop duplicate disposition rows,3377966,2541204,836762
2,keep nature_mutation == Vente,2541204,2389017,152187
3,drop missing valeur_fonciere,2389017,2389017,0
4,drop valeur_fonciere <= 0,2389017,2374309,14708
5,parse date_mutation,2374309,2374309,0
6,cast surface_reelle_bati numeric,2374309,2374309,0
7,cast nombre_pieces_principales numeric,2374309,2374309,0
8,cast surface_terrain numeric,2374309,2374309,0
9,"keep code_type_local in {1,2,3,4}",2374309,2303841,70468


## Missing-value report on cleaned data

In [3]:
missing_report(cleaned).head(20)


,column,missing_count,missing_rate,dtype
0,no_disposition,0,0.00,int64
1,date_mutation,0,0.00,datetime64[s]
2,nature_mutation,0,0.00,str
3,valeur_fonciere,0,0.00,Float64
4,no_voie,0,0.00,float64
5,type_de_voie,0,0.00,str
6,code_voie,0,0.00,str
7,voie,0,0.00,str
8,code_postal,0,0.00,Float64
9,commune,0,0.00,str


In [4]:
save_cleaning_outputs(cleaned, summary)
print('saved cleaned sample and reports')


saved cleaned sample and reports


## Next steps

Continue with EDA (notebooks 03–04) or feature engineering (notebook 05) using the cleaned sample.